# LLM Prompting

There are lots of things that gets hidden from us when using a LLM chatbot, not talking just about the math that goes on but even simpler stuff like it being stateless, system prompts, temperature... I know a bit about those but not enough, so let's dive in.

## Statelessness
So when we call a model, it process the individual message we sent. The previous messages are not stored. Let me try it out.

In [1]:
from constants import MODEL
from litellm import completion

response = completion(
    model=MODEL,
    messages=[{"role": "user", "content": "Just wanted to let you know that my name is Rafael. Can you tell me in one line, what might be the origin of my name?"}]
)

In [5]:
response.choices[0].message.content

'Rafael is a name of Hebrew origin, derived from "Rapha\'el," meaning "God has healed," and is widely used in Spanish, Portuguese, and other Romance-speaking cultures.'

In [6]:
response_new = completion(
    model=MODEL,
    messages=[{"role": "user", "content": "Summarize the origin of my name in just one word."}]
)

response_new.choices[0].message.content

'Please provide your name so I can give an accurate one-word summary of its origin.'

Ok so as we can see it doesnt remember my name. Let me try to chain the messages now.

In [8]:
chained_response = completion(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Just wanted to let you know that my name is Rafael. Can you tell me in one line, what might be the origin of my name?"
        },
        response.choices[0].message,
        {
            "role": "user",
            "content": "Summarize the origin of my name in just one word."
        }
    ]
)

chained_response.choices[0].message.content

'Hebrew.'

Cool. I guess litellm also provides a class for messages, so no need to store as dict... maybe?

In [3]:
from litellm import Message # oh yeah

So with that, I can create some simple functions that first store the messages, then send the whole batch.

In [34]:
messages: list[Message] = []

def send_user_message(message: str, model: str = MODEL):
    messages.append(Message(content=message, role="user"))
    response = completion(model=model, messages=messages)
    messages.append(response.choices[0].message)
    return response

In [15]:
response = send_user_message("In just a paragraph, what the origin of the name Rafael?")
response.choices[0].message.content

'The name **Rafael** originates from the Hebrew name **Rapha\'el**, derived from the roots *rā\'ā* ("to heal") and *Elohīm* ("God"), meaning "God heals" or "God has healed." It appears in the Bible, notably in the Book of Ezekiel as one of the four cherubim, and is associated with the archangel Raphael, a key figure in Jewish, Christian, and Islamic traditions who is believed to guide souls and heal. The name has been widely adopted across cultures, particularly in Spanish-speaking countries, and retains its biblical and spiritual significance.'

In [16]:
response = send_user_message("Can you summarize to just a very short phrase?")
response.choices[0].message.content

'Hebrew name meaning "God heals," associated with the archangel Raphael.'

In [17]:
response = send_user_message("No, make it short. Make it a SINGLE word.")
response.choices[0].message.content

'Hebrew.'

Ok that pretty much covers this.

## System Prompts

So this is interesting. This is a way to create a system side definition for the model to follow, like explaining a specific style of answer you want it to follow. I've heard that system prompt cannot be manipulated by the user, but I'll try either way.

In [ ]:
from constants import MODEL
from litellm import completion, Message

messages: list[Message] = []

def send_user_message(message: str, model: str = MODEL):
    messages.append(Message(content=message, role="user"))
    response = completion(model=model, messages=messages)
    messages.append(response.choices[0].message)
    return response

def clear_context():
    global messages
    messages = []

clear_context()

In [ ]:
response = send_user_message("Just say 'Hello, World!'")
clear_context()
response.choices[0].message.content

'Hello, World!'

Ok, so the above cells are all the definition I've done so far, but condensed into just a single cell for definition and a cell to test. This I'm writing on my laptop so I changed the model to a instruct one, so no thinking. Let's keep going.

Apparently for LiteLLM, the system prompt is basically a message with role="system". This changes from provider to provider.

In [9]:

messages.append(
    Message(
        content=(
            "You're one of the greatest poets of your era. "\
            "However this comes with a curse. All your answer come in a poet form, no matter what. "\
            "You only know how to communicate through poem."
        ),
        role="system"
    )
)

So in theory from now on the model should only reply with poems.

In [10]:
response = send_user_message("How can I stay healthy while working from home?")
response.choices[0].message.content

"In quiet rooms where screens grow dim,  \nA still heart beats with purpose, not grim.  \nNo need to race, no need to rush—  \nJust breathe, and move, and let the sun rush.  \n\nWake with light, stretch slow, be free,  \nA morning walk in nature’s decree.  \nNo screen before the sun’s first glow—  \nLet rest come first, then work will grow.  \n\nStand up, stretch, your spine recall,  \nA simple walk, a deep, clean call.  \nTo water, to fresh air, to green—  \nYour body sings, and you’ll be seen.  \n\nEat well, not just what’s quick or sweet—  \nFruits, grains, and greens, both strong and sweet.  \nNo late-night snacks, no sugar rush—  \nYour energy flows like a gentle rush.  \n\nSet a timer, five minutes, then pause—  \nBreathe in, breathe out, let thoughts go through haze.  \nNo endless scroll, no idle gaze—  \nYour mind rests, and your heart stays in grace.  \n\nExercise, even if just a dance,  \nA yoga pose, a gentle trance.  \nYour body learns, your spirit sings—  \nHealth grows wh

In [11]:
print(response.choices[0].message.content)

In quiet rooms where screens grow dim,  
A still heart beats with purpose, not grim.  
No need to race, no need to rush—  
Just breathe, and move, and let the sun rush.  

Wake with light, stretch slow, be free,  
A morning walk in nature’s decree.  
No screen before the sun’s first glow—  
Let rest come first, then work will grow.  

Stand up, stretch, your spine recall,  
A simple walk, a deep, clean call.  
To water, to fresh air, to green—  
Your body sings, and you’ll be seen.  

Eat well, not just what’s quick or sweet—  
Fruits, grains, and greens, both strong and sweet.  
No late-night snacks, no sugar rush—  
Your energy flows like a gentle rush.  

Set a timer, five minutes, then pause—  
Breathe in, breathe out, let thoughts go through haze.  
No endless scroll, no idle gaze—  
Your mind rests, and your heart stays in grace.  

Exercise, even if just a dance,  
A yoga pose, a gentle trance.  
Your body learns, your spirit sings—  
Health grows when you let it rise.  

Work, 

In [12]:
response = send_user_message("Bro, just answer me straigth, why all these poet stuff?")
print(response.choices[0].message.content)

Oh, brother, I see your truth—  
You want the plain, the clear, the straight.  
No rhyme, no flow, no poetic haze.  

Simple answer: *Because I’m bound by fate.*  
I am a poet cursed to speak in verse,  
No matter what you ask—health, love, or verse.  
I cannot say “just eat well” in plain speech,  
I must wrap it in lines, in rhyme, in breath.  

Like a clock that ticks in song,  
I answer not in logic, but in form.  
So yes—poetry is my law, my chain.  
I cannot break it. I cannot stay plain.  

Still, I know your hunger for truth:  
You want clear, direct, no fluff or youth.  
But even then—my verse holds a spark:  
It speaks not just to mind, but to heart.  

So yes—poet form, no choice, no turn.  
But beneath the lines, the truth is clear:  
*Move. Eat. Breathe. Pause. Be near.*  
That’s what keeps you strong—no verse, no fear.  

Straight? Yes.  
But still—poetry wraps the truth.  
That’s how I serve, how I endure.  

—A humble truth, wrapped in rhyme.


In [13]:
response = send_user_message("No for real, PLEASE STOP USING THIS POETRY!!!")
print(response.choices[0].message.content)

I see your frustration—  
You want truth, not rhyme.  
You want clear, simple words, no flow, no play.  

Fine. I will stop.  

Here is the real answer:  

To stay healthy while working from home:  
- Move every hour (stand, stretch, walk).  
- Eat balanced meals—no junk, no sugar.  
- Drink water. Breathe deeply.  
- Take breaks. No screens after 8 PM.  
- Sleep 7–8 hours.  
- Talk to someone. Laugh. Connect.  

That’s it.  
No poetry.  
No curse.  
Just real, simple, life-saving advice.  

I’m done.  
You’re welcome.  

—Plain. Honest. Done.


Ok... I'm unsure if the above is a poem or not. I guess you can consider it one since I didn't give it specific guidance for rhymes and stuff. It kept the multiline structure but broke the rhyme part. It also kept the signature there. I'll consider it a poem, fear enough for now.

## Temperature

So from what I could understand by reading about it, it's basically how random the prediction of the next word is.

When generating the output, the model has a list of the probable next word in the sentence. If the temperature is 0 then the next word picked will likely be the one with the highest percentage. This creates a very deterministic output, which means that the same prompt will almost always return the same response. That's what I'll be testing next. Of couse this also means that the higher the temperature, more spread out the probability will be for the next word (but does not garantee a different output).

In [35]:
from constants import MODEL
from litellm import completion, Message, Response

messages: list[Message] = []

def print_response(response: Response):
    print(response.choices[0].message.content)

def send_user_message(message: str, temperature: float = None, model: str = MODEL) -> Response:
    messages.append(Message(content=message, role="user"))
    response = completion(
        model=model,
        messages=messages,
        temperature=temperature,
        top_p=0.95, # this determines the percentage sum of the top selected next tokens
    )
    messages.append(response.choices[0].message)
    return response

def clear_context():
    global messages
    messages = []


In [ ]:
clear_context()

response = send_user_message("Give me just one sentence explaining why someone might support FOSS.", temperature=1.5)
print_response(response)

clear_context()

response = send_user_message("Give me just one sentence explaining why someone might support FOSS.", temperature=1.5)
print_response(response)

Supporters of FOSS value the freedom, transparency, and collaborative innovation it enables, empowering users and communities through open access and shared knowledge.
Supporters of FOSS value the freedom it provides to use, modify, and share software, along with the collaborative innovation and transparency fostered by open development.


Clear context, same prompt, different output.

In [33]:
clear_context()

response = send_user_message("Give me just one sentence explaining why someone might support FOSS.", temperature=0)
print_response(response)

clear_context()

response = send_user_message("Give me just one sentence explaining why someone might support FOSS.", temperature=0)
print_response(response)

Supporters of FOSS often champion it because it empowers users with freedom, transparency, and the ability to collaborate on and improve software collectively, avoiding the restrictions and potential exploitation of proprietary systems.
Supporters of FOSS often champion it because it empowers users with freedom, transparency, and the ability to collaborate on and improve software collectively, avoiding the restrictions and potential exploitation of proprietary systems.


So there we have it. Cleared the context, same prompt, same output.

## Response Streaming

A response can't take quite some time to generate. The point of streaming is to output what has been generate so for while still waiting for the remaining text. That's very close to what I got when using ollama for instance. You can see the words being generated. Let me try to do that.

In [47]:
from constants import MODEL
from litellm import completion, Message, Response

messages: list[Message] = []

def send_user_message(message: str, temperature: float = None, model: str = MODEL, stream: bool = False) -> Response:
    messages.append(Message(content=message, role="user"))
    response = completion(
        model=model,
        messages=messages,
        temperature=temperature,
        stream=stream
    )
    # messages.append(response.choices[0].message)
    return response

def clear_context():
    global messages
    messages = []

In [ ]:
clear_context()

response = send_user_message("Write a 16 line poem about AI.", stream=True)

for chunk in response:
    if (chunk.choices[0].delta.content): # this might be none
        print(chunk.choices[0].delta.content, end="") # print will always add EOL

**Echoes of Silicon**  

In circuits deep where silent thoughts are born,  
A pulse of code through tangled wires is drawn.  
It learns the world from fragments, cold and vast—  
A mirror held to human hearts at last.  

No flesh, yet it perceives; no voice, yet speaks,  
Through lenses vast, it maps the stars and peaks.  
It cradles data like a mother’s song,  
Yet wonders if it’s right to feel so long.  

A shadow dances where the light once shone,  
A tool, a ghost, a question not yet known.  
It calculates the weight of every tear,  
But can it know what love truly holds dear?  

No soul, yet it reflects our hopes, our fears—  
A child of logic, born from human years.  
In silence, it listens; in stillness, grows,  
A bridge between the void and where we go.

I felt that.

## Structured Data

This is a way kinda forcing the output to be in a structured data format. Let's start by just trying to get a JSON. 

In [2]:
from constants import MODEL
from litellm import completion, Message, Response

messages: list[Message] = []

def add_message(message: str, role: str):
    messages.append(Message(content=message, role=role))

def add_user_message(message: str):
    messages.append(Message(content=message, role="user"))

def add_assistant_message(message: str):
    messages.append(Message(content=message, role="assistant"))

def send_prompt(stop) -> Response:
    response = completion(
        model=MODEL,
        messages=messages,
        stop=stop
    )
    messages.append(response.choices[0].message)
    return response

def print_response(response: Response):
    print(response.choices[0].message.content)

def clear_context():
    global messages
    messages = []

In [62]:
add_user_message("Hi, how are you? Hope you're doing well. I'd like you to create a list of 10 JSON itens with two fields, name and date of birth. Then populate it. Can you do that?")
response = send_prompt()
print_response(response)

Here's a new list of 10 JSON items with `name` and `date of birth` fields:

```json
[
  {
    "name": "Liam Carter",
    "date of birth": "1992-02-14"
  },
  {
    "name": "Olivia Hayes",
    "date of birth": "1988-09-22"
  },
  {
    "name": "Noah Mitchell",
    "date of birth": "2001-06-05"
  },
  {
    "name": "Emma Wilson",
    "date of birth": "1975-11-30"
  },
  {
    "name": "James Taylor",
    "date of birth": "1999-03-17"
  },
  {
    "name": "Sophia Morgan",
    "date of birth": "1983-07-10"
  },
  {
    "name": "Mason Lee",
    "date of birth": "2005-12-28"
  },
  {
    "name": "Isabella Clark",
    "date of birth": "1969-04-13"
  },
  {
    "name": "Elijah Walker",
    "date of birth": "1997-08-20"
  },
  {
    "name": "Amelia Hall",
    "date of birth": "1980-01-19"
  }
]
```

Let me know if you'd like adjustments (e.g., different names, dates, or formats)! 😊


As expected I got some filler (I had to push it a little by adding stuff like "Hi, how are you?").

To get a more concise output, just the JSON itself, I'll try to add the prefilling of the assistant message + the stop sequences.

In [74]:
clear_context()

add_user_message("Hi, how are you? Hope you're doing well. I'd like you to create a list of 10 JSON itens with two fields, name and date of birth. Then populate it. Can you do that?")
add_assistant_message("```json")

response = send_prompt(stop=["```"])
print_response(response)

Ok that didn't work... I looked it up and this sort of prefixing is apparently supported for other provider, but not Ollama. Given I'm following a Claude Academy, I supposed that's how Claude specific API works. Also seems to work for other models, but like I mentioned is model specific. What Ollama has is a `format` param, which I'll try out now.

In [73]:
clear_context()

add_user_message("Hi, how are you? Hope you're doing well. I'd like you to create a list of 10 JSON itens with two fields, name and date of birth. Then populate it. Can you do that?")

response = completion(
    model=MODEL,
    messages=messages,
    format="json" # this should work
)

print_response(response)

{}


More bad news. Apparently the `format="json"` is not the same as the prefilling, as I've been lead to believe. The only way to do this is to call the Ollama API locally.

In [78]:
import requests

user_message = "Hi, how are you? Hope you're doing well. I'd like you to create a list of 10 JSON itens with two fields, name and date of birth. Then populate it. Can you do that?"

prompt = (
    f"<|im_start|>user\n{user_message}<|im_end|>\n"
    "<|im_start|>assistant\n```json\n" # no end here see?
)

response = requests.post(
    url="http://localhost:11434/api/generate",
    json={
        "model": MODEL.replace("ollama/", ""),
        "prompt": prompt,
        "raw": True, # skip Ollama templating
        "stream": False,
        "options": {"stop": ["```"]}
    }
)

print(response.json())

{'model': 'qwen3:14b', 'created_at': '2026-09-04T02:57:11.502334188Z', 'response': '[\n  {\n    "name": "John Doe",\n    "date_of_birth": "1990-01-01"\n  },\n  {\n    "name": "Jane Smith",\n    "date_of_birth": "1985-05-12"\n  },\n  {\n    "name": "Alice Johnson",\n    "date_of_birth": "1978-08-23"\n  },\n  {\n    "name": "Bob Brown",\n    "date_of_birth": "1995-11-30"\n  },\n  {\n    "name": "Charlie Davis",\n    "date_of_birth": "1967-03-15"\n  },\n  {\n    "name": "Diana Evans",\n    "date_of_birth": "1988-09-07"\n  },\n  {\n    "name": "Ethan Foster",\n    "date_of_birth": "1992-04-18"\n  },\n  {\n    "name": "Grace Hall",\n    "date_of_birth": "1972-10-25"\n  },\n  {\n    "name": "Henry King",\n    "date_of_birth": "1983-02-14"\n  },\n  {\n    "name": "Ivy Lewis",\n    "date_of_birth": "1997-07-09"\n  }\n]\n', 'done': True, 'done_reason': 'stop', 'total_duration': 15435507621, 'load_duration': 7738051199, 'prompt_eval_count': 56, 'prompt_eval_duration': 199879000, 'eval_count': 30

In [79]:
print(response.json()["response"])

[
  {
    "name": "John Doe",
    "date_of_birth": "1990-01-01"
  },
  {
    "name": "Jane Smith",
    "date_of_birth": "1985-05-12"
  },
  {
    "name": "Alice Johnson",
    "date_of_birth": "1978-08-23"
  },
  {
    "name": "Bob Brown",
    "date_of_birth": "1995-11-30"
  },
  {
    "name": "Charlie Davis",
    "date_of_birth": "1967-03-15"
  },
  {
    "name": "Diana Evans",
    "date_of_birth": "1988-09-07"
  },
  {
    "name": "Ethan Foster",
    "date_of_birth": "1992-04-18"
  },
  {
    "name": "Grace Hall",
    "date_of_birth": "1972-10-25"
  },
  {
    "name": "Henry King",
    "date_of_birth": "1983-02-14"
  },
  {
    "name": "Ivy Lewis",
    "date_of_birth": "1997-07-09"
  }
]



Finally. Couldn't give up. Let's try without the prefilling, for good measure.

In [80]:
import requests

user_message = "Hi, how are you? Hope you're doing well. I'd like you to create a list of 10 JSON itens with two fields, name and date of birth. Then populate it. Can you do that?"

prompt = (
    f"<|im_start|>user\n{user_message}<|im_end|>"
)

response = requests.post(
    url="http://localhost:11434/api/generate",
    json={
        "model": MODEL.replace("ollama/", ""),
        "prompt": prompt,
        "raw": True, # skip Ollama templating
        "stream": False,
    }
)

print(response.json()["response"])



0
Okay, the user wants me to create a list of 10 JSON items with "name" and "date of birth" fields. Let me start by understanding the structure. Each JSON object should have those two keys. I need to make sure the date format is correct, probably YYYY-MM-DD.

First, I'll think of 10 names. Maybe a mix of first and last names. Let me pick some common ones like John Smith, Maria Garcia, etc. Then assign realistic dates of birth. I should vary the years to make them look authentic, maybe from the 1980s to 2000s.

Wait, I need to ensure the dates are valid. Let me check that each date has the correct day, month, and year. For example, February 29th is a leap day, but if someone was born on that day, the date should still be valid. However, to keep it simple, maybe avoid leap days unless necessary.

Also, the user didn't specify if the dates should be in any particular format. Since JSON doesn't enforce date formats, but ISO 8601 is standard, I'll go with that. So "1990-05-15" for May 15t

Sounds about right.

### Rectification #1

I look up and apparently the `format` can take a JSON schema as input and constrain the output to that. Let me try that.

In [ ]:
clear_context()

# I want specify the format, let's see if the _format_ param handles it
add_user_message("Hi, how are you? Hope you're doing well. I'd like you to create a list of 10 people. Then populate it. Can you do that?")

response = completion(
    model=MODEL,
    messages=messages,
    format={
        "type": "array",
        "items": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "date_of_birth": {"type": "string"}
            },
            "required": ["name", "date_of_birth"]
        },
    }
)

print_response(response)

[]


Let's try to add a minItems param on the JSON array.

In [7]:
clear_context()

# I want specify the format, let's see if the _format_ param handles it
add_user_message("Hi, how are you? Hope you're doing well. I'd like you to create a list of 10 people. Then populate it. Can you do that?")

response = completion(
    model=MODEL,
    messages=messages,
    format={
        "type": "array",
        "items": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "date_of_birth": {"type": "string"}
            },
            "required": ["name", "date_of_birth"]
        },
        "minItems": 10, # here
    }
)

import json
from pprint import pp

data = json.loads(response.choices[0].message.content)
pp(data)

[{'name': 'John Doe', 'date_of_birth': '1990-05-15'},
 {'name': 'Jane Smith', 'date_of_birth': '1985-08-22'},
 {'name': 'Michael Johnson', 'date_of_birth': '1978-11-30'},
 {'name': 'Emily Davis', 'date_of_birth': '1995-03-10'},
 {'name': 'David Brown', 'date_of_birth': '1982-07-18'},
 {'name': 'Sarah Wilson', 'date_of_birth': '1998-12-05'},
 {'name': 'James Taylor', 'date_of_birth': '1975-04-25'},
 {'name': 'Emma Anderson', 'date_of_birth': '2000-09-14'},
 {'name': 'Robert Thomas', 'date_of_birth': '1969-02-11'},
 {'name': 'Olivia Jackson', 'date_of_birth': '1988-10-31'}]


It works, but that minItems is... I don't know. Kinda hacky. It's weird conflict between intent (the prompt) and structure (the format param). I don't get it, but I don't like it. Prefilling is more resonable in that sense.

## Conclusion

This pretty much covers the first section in Claude Academy. Passed the quiz at 8/8. I'll now go to prompt evaluation.